# Machine Learning Trading Strategy on the CAC 40

**Course:** Machine Learning sous Python  
**Program:** MBA2 Trading et Finance de Marché ESLSCA 
**Academic Year:** 2025/2026  
**Student:** Baptiste DEHAY

## 1. Objective

The objective of this project is to build a trading strategy based on a Feedforward Neural Network applied to the CAC 40 index.

The model aims to predict whether the CAC 40 will increase at J+1 using historical OHLC market data and technical indicators.

The analysis follows four main stages:

1. Data acquisition and feature engineering
2. Neural network training
3. Classification performance evaluation
4. Trading strategy backtest

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    TICKER,
    START_DATE,
    END_DATE,
    TRAIN_RATIO,
    RANDOM_SEED,
    EPOCHS,
    BATCH_SIZE,
    PREDICTION_THRESHOLD,
    PROCESSED_DATA_PATH,
    BACKTEST_DIR,
    FIGURES_DIR,
    METRICS_DIR,
)

from src.data_loader import get_data

from src.features import (
    FEATURE_COLUMNS,
    build_dataset,
)

from src.preprocessing import prepare_data

from src.ml_model import (
    create_model,
    train_model,
    predict_probabilities,
    save_trained_model,
)

from src.evaluation import (
    evaluate_predictions,
    metrics_to_dataframe,
    confusion_matrix_to_dataframe,
    classification_report_to_dataframe,
)

from src.backtest import (
    run_backtest,
    statistics_to_dataframe,
)

from src.visualization import (
    plot_price,
    plot_training_history,
    plot_confusion_matrix,
    plot_prediction_distribution,
    plot_cumulative_returns,
    plot_equity_curve,
    plot_drawdown,
)

In [ ]:
print(f"Ticker: {TICKER}")
print(f"Period: {START_DATE} to {END_DATE}")
print(f"Training ratio: {TRAIN_RATIO:.0%}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Prediction threshold: {PREDICTION_THRESHOLD}")

## 2. CAC 40 Market Data

The financial instrument used in this study is the CAC 40 index, identified by the Yahoo Finance ticker `^FCHI`.

Daily market data are collected from January 1, 2017 to January 1, 2022.

In [ ]:
data = get_data()

data.head()

In [ ]:
print(f"Number of observations: {len(data)}")
print(f"First observation: {data.index.min()}")
print(f"Last observation: {data.index.max()}")

In [ ]:
data.info()

In [ ]:
data.isna().sum()

In [ ]:
data.describe()

In [ ]:
plot_price(data)

## 3. Feature Engineering

In [ ]:
dataset = build_dataset(data)

dataset.head()

PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

dataset.to_csv(
    PROCESSED_DATA_PATH
)

print(
    f"Processed dataset saved to: "
    f"{PROCESSED_DATA_PATH}"
)

In [ ]:
print(f"Number of observations: {len(dataset)}")
print(f"Number of columns: {dataset.shape[1]}")

In [ ]:
FEATURE_COLUMNS

In [ ]:
dataset[FEATURE_COLUMNS].head()

In [ ]:
dataset[FEATURE_COLUMNS].describe()

In [ ]:
dataset[FEATURE_COLUMNS].isna().sum()

## 4. Target Variable

In [ ]:
dataset["Label"].value_counts()

In [ ]:
dataset["Label"].value_counts(normalize=True)

In [ ]:
pd.DataFrame(
    {
        "Count": dataset["Label"].value_counts().sort_index(),
        "Proportion": dataset["Label"].value_counts(
            normalize=True
        ).sort_index(),
    }
)

## 5. Training and Test Data

In [ ]:
prepared = prepare_data(
    dataset,
    train_ratio=TRAIN_RATIO,
)

In [ ]:
X_train = prepared["X_train"]
X_test = prepared["X_test"]

y_train = prepared["y_train"]
y_test = prepared["y_test"]

X_train_scaled = prepared["X_train_scaled"]
X_test_scaled = prepared["X_test_scaled"]

scaler = prepared["scaler"]

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
print(
    "Training period:",
    X_train.index.min(),
    "to",
    X_train.index.max(),
)

print(
    "Test period:",
    X_test.index.min(),
    "to",
    X_test.index.max(),
)

In [ ]:
print(
    "X_train_scaled:",
    X_train_scaled.shape
)

print(
    "X_test_scaled:",
    X_test_scaled.shape
)

In [ ]:
np.isnan(X_train_scaled).sum()

In [ ]:
np.isnan(X_test_scaled).sum()

## 6. Feedforward Neural Network

A Feedforward Neural Network is used as the supervised learning algorithm.

The network receives the seven standardized features as inputs and follows the architecture specified in the assignment:

- Input Layer: 7 features
- Hidden Layer 1: 512 neurons, ELU activation
- Dropout: 15%
- Hidden Layer 2: 256 neurons, ReLU activation
- Dropout: 15%
- Hidden Layer 3: 128 neurons, ELU activation
- Dropout: 15%
- Hidden Layer 4: 32 neurons, ReLU activation
- Dropout: 15%
- Output Layer: 1 neuron, Sigmoid activation

The model is compiled using the Adam optimizer, binary cross-entropy loss and accuracy as the evaluation metric.

In [ ]:
input_dim = X_train_scaled.shape[1]

print(f"Number of input features: {input_dim}")

In [ ]:
model = create_model(
    input_dim=input_dim
)

In [ ]:
model.summary()

In [ ]:
print(f"Input dimension: {model.input_shape[1]}")
print(f"Output dimension: {model.output_shape[1]}")
print(f"Loss function: {model.loss}")
print(f"Optimizer: {model.optimizer.__class__.__name__}")

In [ ]:
for layer in model.layers:
    print(
        layer.__class__.__name__,
        layer.name,
        getattr(layer, "units", ""),
        getattr(layer, "activation", ""),
        getattr(layer, "rate", ""),
    )

The output layer contains a single neuron with a sigmoid activation function. Therefore, the network produces a probability between 0 and 1.

A probability greater than or equal to 0.5 is subsequently classified as `1`, corresponding to the positive class, whereas a probability below 0.5 is classified as `0`.

Dropout regularization is applied after each hidden layer in order to reduce overfitting.

## 7. Model Training

The Feedforward Neural Network is trained on the standardized training dataset.

In accordance with the assignment specifications, the model is trained for 25 epochs using a batch size of 64 observations.

The training data remain chronologically ordered throughout the training process.

In [ ]:
history = train_model(
    model=model,
    X_train=X_train_scaled,
    y_train=y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

In [ ]:
history.history.keys()

In [ ]:
training_history = pd.DataFrame(
    history.history
)

training_history.index = (
    training_history.index + 1
)

training_history.index.name = "Epoch"

training_history

In [ ]:
print(
    f"Final training accuracy: "
    f"{history.history['accuracy'][-1]:.4f}"
)

print(
    f"Final training loss: "
    f"{history.history['loss'][-1]:.4f}"
)

In [ ]:
plot_training_history(
    history,
    save_path=FIGURES_DIR / "training_history.png",
)

In [ ]:
model_path = save_trained_model(
    model
)

print(
    f"Model saved to: {model_path}"
)

The evolution of the training loss and accuracy provides a first indication of the convergence of the neural network.

However, training accuracy alone is not sufficient to assess predictive performance. The model must subsequently be evaluated on the previously unseen test dataset.

The next section therefore applies the trained network to the test observations in order to generate J+1 probability forecasts.

## 8. Test Set Predictions

The trained neural network is now applied to the standardized test dataset.

For each observation, the sigmoid output represents the estimated probability of belonging to class `1`.

The binary prediction is obtained using a classification threshold of 0.5:

- Probability < 0.5 → Class 0
- Probability ≥ 0.5 → Class 1

In [ ]:
y_proba = predict_probabilities(
    model,
    X_test_scaled,
)

print(y_proba.shape)

In [ ]:
y_pred = (
    y_proba >= PREDICTION_THRESHOLD
).astype(int)

print(y_pred.shape)

In [ ]:
prediction_results = pd.DataFrame(
    {
        "Actual": y_test.values,
        "Probability": y_proba,
        "Predicted": y_pred,
    },
    index=X_test.index,
)

prediction_results.head()

In [ ]:
prediction_results.tail()

In [ ]:
prediction_results["Predicted"].value_counts()

In [ ]:
prediction_results["Predicted"].value_counts(
    normalize=True
)

In [ ]:
pd.DataFrame(
    {
        "Actual": y_test.value_counts().sort_index(),
        "Predicted": prediction_results[
            "Predicted"
        ].value_counts().sort_index(),
    }
).fillna(0).astype(int)

In [ ]:
prediction_results["Probability"].describe()

In [ ]:
plot_prediction_distribution(
    y_proba,
    threshold=PREDICTION_THRESHOLD,
    save_path=FIGURES_DIR / "prediction_distribution.png",
)

The predicted probabilities provide more information than the binary classifications alone.

The distribution of probabilities makes it possible to assess whether the neural network produces strongly differentiated forecasts or whether most predictions remain concentrated around the 0.5 decision threshold.

The predictive quality of these classifications must now be evaluated against the actual test labels using the confusion matrix, precision, recall and F1-score.

## 9. Classification Performance Evaluation

The predictions obtained on the test dataset are now compared with the actual target values.

The predictive performance of the neural network is assessed using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix
- Classification report

In [ ]:
evaluation = evaluate_predictions(
    y_true=y_test,
    probabilities=y_proba,
    threshold=PREDICTION_THRESHOLD,
)

In [ ]:
metrics_df = metrics_to_dataframe(
    evaluation["metrics"]
)

metrics_df

In [ ]:
print(
    f"Accuracy: "
    f"{evaluation['metrics']['Accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{evaluation['metrics']['Precision']:.4f}"
)

print(
    f"Recall: "
    f"{evaluation['metrics']['Recall']:.4f}"
)

print(
    f"F1 Score: "
    f"{evaluation['metrics']['F1 Score']:.4f}"
)

In [ ]:
confusion_df = confusion_matrix_to_dataframe(
    y_test,
    evaluation["predictions"],
)

confusion_df

In [ ]:
plot_confusion_matrix(
    evaluation["confusion_matrix"],
    save_path=FIGURES_DIR / "confusion_matrix.png",
)

In [ ]:
classification_df = (
    classification_report_to_dataframe(
        y_test,
        evaluation["predictions"],
    )
)

classification_df

In [ ]:
print(
    evaluation["classification_report"]
)

In [ ]:
majority_class_accuracy = (
    y_test.value_counts(normalize=True).max()
)

model_accuracy = evaluation[
    "metrics"
]["Accuracy"]

print(
    f"Majority-class baseline accuracy: "
    f"{majority_class_accuracy:.4f}"
)

print(
    f"Neural network accuracy: "
    f"{model_accuracy:.4f}"
)

print(
    f"Accuracy improvement: "
    f"{model_accuracy - majority_class_accuracy:.4f}"
)

In [ ]:
tn, fp, fn, tp = (
    evaluation["confusion_matrix"].ravel()
)

print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

Accuracy measures the overall proportion of correctly classified observations.

Precision measures the proportion of predicted positive observations that are actually positive.

Recall measures the proportion of actual positive observations that are correctly identified by the model.

The F1-score provides a harmonic balance between precision and recall.

Because the target classes are not equally represented, the accuracy metric should not be interpreted in isolation. The confusion matrix, precision, recall and F1-score provide a more complete assessment of the model's ability to identify the positive class.

The statistical classification performance does not necessarily imply that the model generates a profitable trading strategy.

The economic value of the predictions must therefore be evaluated separately through a backtest based on the predicted J+1 signals.

## 10. Trading Strategy Backtest

The economic relevance of the neural network predictions is evaluated through a long-only trading strategy.

For each observation of the test period:

- Prediction = 1 → Long position on the CAC 40 from J to J+1
- Prediction = 0 → No position

The strategy return is therefore determined by the predicted signal and the subsequent CAC 40 return.

The backtest evaluates the number of trades, winning and losing trades, P&L, cumulative return and additional risk-adjusted performance statistics.

In [ ]:
test_prices = dataset.loc[
    X_test.index,
    "Close",
]

print(
    f"Number of test prices: "
    f"{len(test_prices)}"
)

print(
    f"Number of predictions: "
    f"{len(y_proba)}"
)

In [ ]:
backtest_df, backtest_stats = run_backtest(
    prices=test_prices,
    predictions=y_proba,
    threshold=PREDICTION_THRESHOLD,
)

In [ ]:
backtest_df.head()

In [ ]:
backtest_df.tail()

In [ ]:
backtest_results = statistics_to_dataframe(
    backtest_stats
)

backtest_results

In [ ]:
BACKTEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
training_history.to_csv(
    METRICS_DIR / "training_history.csv"
)

prediction_results.to_csv(
    METRICS_DIR / "test_predictions.csv"
)

metrics_df.to_csv(
    METRICS_DIR / "classification_metrics.csv"
)

confusion_df.to_csv(
    METRICS_DIR / "confusion_matrix.csv"
)

classification_df.to_csv(
    METRICS_DIR / "classification_report.csv"
)

backtest_df.to_csv(
    BACKTEST_DIR / "backtest_results.csv"
)

backtest_results.to_csv(
    BACKTEST_DIR / "backtest_statistics.csv"
)

In [ ]:
print("Metrics:")
for path in sorted(METRICS_DIR.glob("*")):
    print(path.name)

print("\nBacktest:")
for path in sorted(BACKTEST_DIR.glob("*")):
    print(path.name)

In [ ]:
print(
    f"Initial capital: "
    f"{backtest_stats['Initial Capital']:,.2f}"
)

print(
    f"Final capital: "
    f"{backtest_stats['Final Capital']:,.2f}"
)

print(
    f"P&L: "
    f"{backtest_stats['P&L']:,.2f}"
)

print(
    f"Cumulative return: "
    f"{backtest_stats['Cumulative Return']:.2%}"
)

print(
    f"Benchmark return: "
    f"{backtest_stats['Benchmark Return']:.2%}"
)

In [ ]:
print(
    f"Total trades: "
    f"{backtest_stats['Total Trades']}"
)

print(
    f"Winning trades: "
    f"{backtest_stats['Winning Trades']}"
)

print(
    f"Losing trades: "
    f"{backtest_stats['Losing Trades']}"
)

print(
    f"Flat trades: "
    f"{backtest_stats['Flat Trades']}"
)

print(
    f"Win rate: "
    f"{backtest_stats['Win Rate']:.2%}"
)

In [ ]:
print(
    f"Average trade return: "
    f"{backtest_stats['Average Trade Return']:.4%}"
)

print(
    f"Best trade: "
    f"{backtest_stats['Best Trade']:.4%}"
)

print(
    f"Worst trade: "
    f"{backtest_stats['Worst Trade']:.4%}"
)

print(
    f"Annualized volatility: "
    f"{backtest_stats['Annualized Volatility']:.2%}"
)

print(
    f"Sharpe ratio: "
    f"{backtest_stats['Sharpe Ratio']:.4f}"
)

print(
    f"Maximum drawdown: "
    f"{backtest_stats['Maximum Drawdown']:.2%}"
)

print(
    f"Market exposure: "
    f"{backtest_stats['Market Exposure']:.2%}"
)

In [ ]:
plot_cumulative_returns(
    backtest_df,
    save_path=FIGURES_DIR / "cumulative_returns.png"
)

In [ ]:
plot_equity_curve(
    backtest_df,
    save_path=FIGURES_DIR / "equity_curve.png",
)

In [ ]:
plot_drawdown(
    backtest_df,
    save_path=FIGURES_DIR / "drawdown.png",
)

In [ ]:
performance_comparison = pd.DataFrame(
    {
        "Strategy": [
            backtest_stats[
                "Cumulative Return"
            ],
            backtest_stats[
                "Annualized Volatility"
            ],
            backtest_stats[
                "Sharpe Ratio"
            ],
            backtest_stats[
                "Maximum Drawdown"
            ],
        ],
    },
    index=[
        "Cumulative Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
    ],
)

performance_comparison

In [ ]:
excess_performance = (
    backtest_stats["Cumulative Return"]
    - backtest_stats["Benchmark Return"]
)

print(
    f"Strategy cumulative return: "
    f"{backtest_stats['Cumulative Return']:.2%}"
)

print(
    f"CAC 40 cumulative return: "
    f"{backtest_stats['Benchmark Return']:.2%}"
)

print(
    f"Excess performance: "
    f"{excess_performance:.2%}"
)

The backtest translates the statistical predictions of the neural network into economically observable results.

The win rate measures the proportion of profitable trades, while cumulative return measures the total compounded performance of the strategy over the test period.

The Sharpe ratio provides an indication of risk-adjusted performance, whereas maximum drawdown measures the largest decline of the strategy from a previous equity peak.

The comparison with the CAC 40 buy-and-hold benchmark makes it possible to determine whether the machine-learning signals generated economic value beyond simple passive exposure to the index.

## 11. Results Interpretation and Discussion

The neural network demonstrates meaningful predictive ability on the out-of-sample test dataset.

The model achieves an accuracy of 73.81%, compared with a majority-class baseline accuracy of 60.71%, corresponding to an improvement of 13.10 percentage points.

For the positive class, the model achieves a precision of 64.60%, a recall of 73.74% and an F1-score of 68.87%. These results indicate that the neural network is able to identify a substantial proportion of positive J+1 signals while maintaining a moderate false-positive rate.

### 11.1 Trading Performance

The trading strategy generates a cumulative return of 16.20% over the test period, increasing the initial capital from 10,000 to 11,620.14.

A total of 113 long trades are executed, of which 73 are profitable and 40 are unprofitable, corresponding to a win rate of 64.60%.

The average return per trade is 0.1344%, while the best and worst trades generate respectively 1.4478% and -1.8572%.

### 11.2 Comparison with the CAC 40

The CAC 40 buy-and-hold benchmark produces a cumulative return of 26.68% over the same period, compared with 16.20% for the machine-learning strategy.

The strategy therefore underperforms the benchmark by 10.48 percentage points in absolute return terms.

However, this comparison must be interpreted cautiously because the two strategies do not have the same market exposure. The machine-learning strategy is invested only 45.02% of the time, whereas the buy-and-hold benchmark remains continuously exposed to the CAC 40.

### 11.3 Risk-Adjusted Performance

The strategy exhibits an annualized volatility of 5.87% and a maximum drawdown of -2.57%.

Its Sharpe ratio reaches 2.60 when assuming a zero risk-free rate.

These results suggest that, although the strategy does not outperform the CAC 40 in absolute cumulative return, it generates positive returns with relatively limited volatility and drawdown.

The machine-learning signal therefore appears more attractive from a risk-adjusted perspective than the comparison of cumulative returns alone would suggest.

### 11.4 Limitations

Several limitations must be considered when interpreting the results.

First, the out-of-sample backtest covers only approximately one year of market data. The observed performance may therefore depend on the specific market conditions prevailing during 2021.

Second, transaction costs are assumed to be zero. Consequently, the reported strategy return represents a gross theoretical performance and would be reduced by brokerage fees, bid-ask spreads and other execution costs in a real trading environment.

Third, the classification threshold is fixed at 0.5 and has not been optimized.

Finally, the neural network is evaluated using a single chronological train-test split. Additional robustness tests, such as walk-forward validation or multiple out-of-sample periods, would be necessary before drawing conclusions regarding the stability of the strategy.

## 12. Conclusion

This project developed and evaluated a Feedforward Neural Network for predicting the CAC 40 target at J+1 using historical price information and technical indicators.

On the out-of-sample test dataset, the neural network achieved an accuracy of 73.81%, exceeding the majority-class baseline of 60.71%. The model also obtained a precision of 64.60%, a recall of 73.74% and an F1-score of 68.87%.

The resulting trading strategy generated a positive cumulative return of 16.20%, with a win rate of 64.60%, an annualized volatility of 5.87%, a Sharpe ratio of 2.60 and a maximum drawdown of -2.57%.

However, the strategy underperformed the CAC 40 buy-and-hold benchmark in absolute return terms, as the benchmark gained 26.68% over the same period. This difference must be considered alongside the substantially lower market exposure of the machine-learning strategy, which was invested only 45.02% of the time.

Overall, the results suggest that the neural network provides economically relevant predictive information, although additional out-of-sample testing, transaction-cost modelling and walk-forward validation would be required before considering the strategy sufficiently robust for practical implementation.

CREDITS : Baptiste DEHAY (WAVETROPY LABS ; www.wavetropy.com)